# Euronext universe pipeline — v2 CSV export

Ce notebook lance le scraper corrigé. La correction principale : utiliser l'export CSV public Euronext au lieu de l'endpoint JSON/DataTables qui peut renvoyer 0 ligne selon la session.

Sorties attendues :

- `data/euronext_universe/euronext_current_universe.csv`
- `data/euronext_universe/euronext_candidate_universe.csv`
- `data/euronext_universe/euronext_universe_timeseries.csv`
- `data/euronext_universe/euronext_universe_for_load_user_universe.csv`
- `data/euronext_universe/euronext_universe_summary.csv`


In [ ]:
# Dépendances à installer une seule fois si besoin
# %pip install -U pandas requests beautifulsoup4 tqdm yfinance pyarrow nbformat

In [1]:
from pathlib import Path
import os
import sys
import shutil
import importlib
import pandas as pd

# Résoudre la racine du projet : quand le notebook est dans notebooks/,
# .parent remonte au repo root qui contient euronext_simstock/
PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / 'euronext_simstock').exists():
    PROJECT_ROOT = Path.cwd()  # fallback si ouvert depuis la racine

PROJECT_ROOT


In [2]:
# Se placer dans le projet et rendre les imports locaux disponibles
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Working directory:', Path.cwd())

## Importer le scraper

Le script `euronext_universe_timeseries_scraper.py` se trouve à la racine du repo.
Il est automatiquement disponible une fois que `PROJECT_ROOT` est dans `sys.path`.


In [3]:
# Le scraper est déjà en place à la racine du repo.
# Aucune copie nécessaire.


In [4]:
# Import du scraper
import euronext_universe_timeseries_scraper as scraper
importlib.reload(scraper)

print('Module chargé depuis:', scraper.__file__)
print('URL export CSV:', getattr(scraper, 'EURONEXT_STOCK_DOWNLOAD_URL', 'NON TROUVÉE'))


## Test rapide : télécharger uniquement l'univers courant

Cette cellule permet d'isoler le problème Euronext avant de lancer tous les téléchargements Yahoo. Elle doit afficher un nombre de lignes > 0.

In [5]:
cfg = scraper.RequestConfig(timeout=45, min_delay=0.5, max_retries=5)
session = scraper.make_session(cfg)

current = scraper.scrape_current_euronext_universe(session=session, cfg=cfg, page_size=100)
print('Rows:', len(current))
current.head()

In [6]:
# Sauvegarde du snapshot courant, utile pour relancer sans retélécharger Euronext
OUT_DIR = PROJECT_ROOT / 'data' / 'euronext_universe'
OUT_DIR.mkdir(parents=True, exist_ok=True)
current_path = OUT_DIR / 'euronext_current_universe.csv'
current.to_csv(current_path, index=False)
print(current_path)

## Lancer le pipeline complet

Le pipeline va ensuite vérifier l'historique Yahoo/yfinance entre 2019 et fin 2025 pour construire la série temporelle. Mets `include_notices=True` pour tenter d'ajouter des candidats delistés via les cash notices, mais cela peut être long.

In [7]:
from argparse import Namespace

START_DATE = '2019-01-01'
END_DATE = '2025-12-31'

args = Namespace(
    start=START_DATE,
    end=END_DATE,
    out_dir=str(OUT_DIR),
    include_notices=False,      # mets True si tu veux scraper aussi les notices de delisting
    current_csv=str(current_path),
    force=True,                 # force la reconstruction des CSV de sortie
    page_size=100,
    notice_page_size=50,
    max_notice_pages=1500,
    max_workers=4,
    min_observations=1,
    timeout=45,
    min_delay=0.5,
    max_retries=5,
)

scraper.run_pipeline(args)

## Vérifier les sorties

In [8]:
files = [
    'euronext_current_universe.csv',
    'euronext_candidate_universe.csv',
    'euronext_universe_timeseries.csv',
    'euronext_universe_for_load_user_universe.csv',
    'euronext_universe_summary.csv',
]

for f in files:
    p = OUT_DIR / f
    print(f'{f}:', 'OK' if p.exists() else 'MANQUANT', p)

summary = pd.read_csv(OUT_DIR / 'euronext_universe_summary.csv')
summary.head()

In [9]:
# Fichier directement compatible avec load_user_universe
load_path = OUT_DIR / 'euronext_universe_for_load_user_universe.csv'
load_df = pd.read_csv(load_path)
print(load_df.shape)
load_df.head()

In [ ]:
# Optionnel : charger avec ton universe.py
# À utiliser si le package euronext_simstock est bien importable depuis PROJECT_ROOT.
from euronext_simstock.universe import load_user_universe

universe = load_user_universe(load_path)
print(universe.shape)
universe.head()